# Evaluate (Lean): The App's Rankers on Real Career Moves

PredictLeads × TripleTen externship · Jacob Colombo

## Purpose

Scores the rankers the app actually serves on **real career moves, with no labelling** (`PROJECT.md`, How we keep it honest). `ML_Fit` scored its own methods this way: for Profiles' career changers we know the role each one actually moved into, and a method is scored on whether that role is in its top 3 and top 5 from the person's background alone. This notebook puts the app's three rankers on the same answer key:

| Ranker | What it is | Where |
|:--|:--|:--|
| `rrf` | Jesse's text match: TF-IDF and BGE fused by reciprocal rank | `app/backend/rrf_ranker.py` |
| `mlrank` | Jacob's app: ML_Rank's priority queue over the text match | `app/backend/ml_rank_ranker.py` |
| combined | The combined model (**v2**): ML_Fit's role fit as the fit factor of ML_Rank's queue, over the text match, on the scope's listings. v2 blends the role choice with the role mix of the person's text matches and puts a stated desired role first (`combined/evaluate.py`) | `combined/` |

**How a listing ranker is scored on roles:** each ranker ranks the serving corpus for a person; its top 20 listings (the app's first page) become a role ranking, a role scoring how many of those listings are in it (ties broken by the best position). `fit_eval.evaluate` then scores it exactly as `ML_Fit` 6b did.

**The series** (each notebook hands its final table to the next):

| Notebook | Question | Hands off |
|:--|:--|:--|
| EDA_SDA_1 | What's in the data, how does tech compare to other fields, and what's in scope? | Scope table, descriptions, occupation summary, company activity |
| EDA_SDA_2 | Which in-scope listings do we keep, tag or hide? | Scope table, non-text features |
| EDA_SDA_3 | What do the descriptions say? Skills mapped to O*NET | Scope table + text features |
| EDA_SDA_4 | Which features hold up? Statistics, confounders, the adjusted Cox check | Modeling-ready table |
| TimeSeries | When do listings appear and close? | Timing features |
| Profiles | Who really changed into or within tech or math, and how? Career-switch rules on the public résumé data (frozen) | The 2,813 career changers, and the table ML_Fit trains on |
| ML_Fit | Which roles and listings fit a career changer? Keyword → TF-IDF → BGE → RRF, skills-based matching, a trained layer on real career moves | Fit scores, role discovery, the compass |
| ML_Rank | How should listings be ranked? The listing timing model, combined with fit into the priority queue | The priority queue algorithm |
| **Evaluate** | **How well do the app's rankers do on real career moves? `rrf`, `mlrank` and the combined model** | **The ranker comparison** |

**Clarifying questions**
- Do the app's rankers find the role real career changers moved into more often than the most-common-role baseline?
- Does adding ML_Rank's queue (`mlrank`) or the trained layer's role fit (combined) change that?
- How close does each come to `ML_Fit`'s trained layer (81.7% at top 3), the best the role-level evidence offers?

**What it can't answer:** whether a listing is a good one to apply to (that's `ML_Rank`'s backtest, 8d–8e), and the roles Profiles has no moves into (sales engineers, business and market research analysts: no answer key). **It also can't see half of what v2 does:** Profiles' backgrounds are résumé titles, programs and dates — exactly the trained layer's own inputs — and carry no desired role, so v2's two changes (blending in the text match's role mix, putting a desired role first) are scored here with their cost and none of their benefit. A number from this notebook favours v1 by construction; section 5 says what follows from that.

### About this Lean draft

A **replica** of `Evaluate`, threaded into the Lean family rather than rewritten from its question (`docs/lean_conventions.md`, Replicas). The formatting helpers are gone (rule 1): its 6 `show()` calls are plain `print()` + `display()`.

**Nothing to move.** Unlike the other replicas, this notebook saves nothing - it loads the rankers the app serves and scores them, so there was no hand-off path that could overwrite an original's. Its reads go to the app's own artifacts through `combined/evaluate.py`, which is the point of the notebook.

**Open.** Its 77 lines of code are thin because the work lives in `combined/evaluate.py`, `src/fit_eval.py` and `src/rank_model.py`, and the company-clustered bootstrap behind its intervals (`rm.clustered_bootstrap_diff`) has no teaching doc - the same question rule 9 asks of any shared method: does it change a decision? Here it does, since the decision rule in 4c rests on it.

**Run 2026-10-28, on the original hand-offs.** It reads the same artifacts as `Evaluate`: the serving corpus and combined bundle under `data/clean/combined_v1/`, `data/clean/ml_fit/` and `fit_scores.parquet`, `data/clean/ml_rank/` and `data/profiles/`, not the Lean copies under `data/clean/lean/`. Every hit rate, interval and E1-E4 line therefore reproduces `Evaluate`'s 2026-10-23 run on combined v2 exactly; only the timings differ (machine load). The findings below are written from this run.

## 1. Setup

In [ ]:
# 1a Libraries and the shared helpers; combined/ sits at the repository root
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = next(d for d in [Path.cwd(), *Path.cwd().parents] if (d / "src" / "eda_helpers.py").exists())
sys.path.insert(0, str(ROOT))         # analysis/, for `src`
sys.path.insert(0, str(ROOT.parent))  # the repository root, for `combined`
from src.eda_helpers import BLUE, ORANGE, AQUA, MUTED, finish, use_chart_style
# chart setup and colours only: show() is gone, per docs/lean_conventions.md rule 1
from src import fit_eval as fe
from src import rank_model as rm
from combined import evaluate as ev
use_chart_style()

## 1c. Hypotheses and decision rule (written before any ranker is run)

The test people are `ML_Fit`'s held-out 20%, scored **once**, at the end; nothing here is tuned. **One rule decides E1 to E4:** the first method beats the second at top 3, and the 95% bootstrap interval for the per-person difference in hits (people resampled; `rm.clustered_bootstrap_diff` with each person as their own cluster) excludes zero. The size of the difference is reported either way.

| # | Hypothesis | Compared with | Supported when |
|:--|:--|:--|:--|
| E1 | `rrf` finds the moved-into role more often than the baseline | Most common role (baseline), top 3 | Difference positive, interval excludes 0 |
| E2 | The combined model beats the baseline | Same | Same |
| E3 | The combined model beats `rrf` | `rrf`, top 3 | Same |
| E4 | The combined model beats `mlrank` | `mlrank`, top 3 | Same |

**Fixed before the run:** the top 20 listings per person; a role's score is its count among them, ties broken by the best position; listings outside `ML_Fit`'s 8 roles don't count; for moves within tech the current role is left out, as in `ML_Fit`. **Secondary, not deciding:** top 5, the pools, and the top 10 listings instead of 20.

**Anchor (a check, not a hypothesis):** the trained layer scored from `ML_Fit`'s saved hand-off must give the same hits as `ML_Fit` 6b on the same people, or the setup is wrong and nothing below is used.

## 2. The career changers and the split

`ML_Fit` 2a's people and split (`fit_eval.split_people`, seed 0): the development people give the baseline, the test people are scored.

In [ ]:
# 2a The people and the split (counts only)
people, dev, test = ev.load_people()
test_people = people.loc[test]
print("Career changers by pool (people)")
display(pd.DataFrame({
    "development": people.loc[dev].pool.value_counts(), "test (held out)": test_people.pool.value_counts()}).fillna(0).astype(int))
print(f"{len(test_people):,} test people, {len(dev):,} development people; the baseline uses the development people's roles")

## 3. The rankers and the anchor

One RRF ranker over the serving corpus is shared by all three (loading it fits TF-IDF over the corpus: a few minutes).

In [ ]:
# 3a Load the three rankers over the serving corpus, and the saved trained layer
import time

start = time.time()
apps = ev.AppRankers()
print(f"loaded in {(time.time() - start) / 60:.1f} min: {len(apps.base.job_ids):,} serving listings, "
      f"{int(apps.listings.hidden.sum()):,} hidden by default")

In [ ]:
# 3b Anchor: the trained layer from the saved hand-off gives ML_Fit 6b's hits on the same people
probs = apps.trained_layer(test_people)
fit_scores = pd.read_parquet(ev.DATA_DIR / "clean" / "fit_scores.parquet")
fit_6b = fit_scores.loc[test_people.index, [f"fit: {r}" for r in fe.ROLES]].to_numpy()
exclude = test_people.from_bucket.where(test_people.from_bucket.notna(), None).tolist()
targets = test_people.target_bucket.tolist()
same = {k: float((fe.top_k_hits(probs, targets, exclude, k) == fe.top_k_hits(fit_6b, targets, exclude, k)).mean()) for k in (3, 5)}
print(f"same hit or miss as ML_Fit 6b: {same[3]:.1%} of people at top 3, {same[5]:.1%} at top 5")
assert same[3] == 1.0, "the anchor fails: the saved hand-off doesn't reproduce ML_Fit 6b at top 3, so stop here"

## 4. The test (run once)

Each ranker ranks the whole serving corpus for each test person, from their pre-switch background (titles and programs, as in `ML_Fit`). About a second or two per person per ranker.

In [ ]:
# 4a Run the three rankers on every test person (counts and timings only)
scores, seconds, tops = ev.score(test_people, apps.rankers(test_people, probs=probs), apps.listings.roles, fe.ROLES)
scores["trained layer (ML_Fit, anchor)"] = probs
print("Seconds per person")
display(pd.Series(seconds).to_frame("seconds"))

In [ ]:
# 4b Hit rates at top 3 and 5, overall and per pool, with 95% bootstrap intervals (fit_eval.evaluate, as ML_Fit 6b)
table = fe.evaluate(scores, test_people, people.loc[dev].target_bucket.tolist())
overall = table[table.pool.eq("all career changers")].pivot(index="method", columns="k", values=["hit_rate", "ci_low", "ci_high"])
fmt = lambda k: overall.apply(lambda r: f"{r[('hit_rate', k)]:.1%} [{r[('ci_low', k)]:.1%}, {r[('ci_high', k)]:.1%}]", axis=1)
print("4b Right role in the top k, all test people (95% interval)")
display(pd.DataFrame({"top 3": fmt(3), "top 5": fmt(5)}))
by_pool = table[table.k.eq(3) & table.pool.ne("all career changers")].pivot(index="method", columns="pool", values="hit_rate")
print("4b Top 3, by pool")
display(by_pool.map("{:.1%}".format))

In [ ]:
# 4c The decision rule: per-person differences in top-3 hits, people resampled (E1-E4)
hits = {m: fe.top_k_hits(s, targets, exclude, 3) for m, s in scores.items()}
hits["most common role (baseline)"] = fe.top_k_hits(fe.baseline_scores(len(test_people), people.loc[dev].target_bucket.tolist()), targets, exclude, 3)
pairs = {"E1 rrf vs. baseline": ("rrf (Jesse's app)", "most common role (baseline)"),
         "E2 combined vs. baseline": ("combined model", "most common role (baseline)"),
         "E3 combined vs. rrf": ("combined model", "rrf (Jesse's app)"),
         "E4 combined vs. mlrank": ("combined model", "mlrank (Jacob's app)")}
rows = {name: rm.clustered_bootstrap_diff(hits[a], hits[b], clusters=test_people.index) for name, (a, b) in pairs.items()}
print("4c Difference in top-3 hit rate (first minus second), 95% interval")
display(pd.DataFrame(rows).T.assign(
    difference=lambda d: d.difference.map("{:+.1%}".format), ci_low=lambda d: d.ci_low.map("{:+.1%}".format),
    ci_high=lambda d: d.ci_high.map("{:+.1%}".format))[["difference", "ci_low", "ci_high", "supported"]])

In [ ]:
# 4d Secondary: the top 10 listings instead of 20 (the same runs, rescored)
top10 = {m: ev.role_scores(t, fe.ROLES, n=10) for m, t in tops.items()}
t10 = fe.evaluate(top10, test_people, people.loc[dev].target_bucket.tolist())
print("4d Top 3 from each ranker's top 10 listings (all test people)")
display(t10[t10.k.eq(3) & t10.pool.eq("all career changers")].set_index("method").hit_rate.map("{:.1%}".format).to_frame("top 3"))

In [ ]:
# 4e The comparison chart: text match alone -> plus the queue -> plus role fit, against the baseline and the trained layer
order = ["most common role (baseline)", "rrf (Jesse's app)", "mlrank (Jacob's app)", "combined model", "trained layer (ML_Fit, anchor)"]
t3 = table[table.k.eq(3) & table.pool.eq("all career changers")].set_index("method").reindex(order)
fig, ax = plt.subplots(figsize=(9, 3.6))
colors = [MUTED, BLUE, BLUE, ORANGE, AQUA]
ax.barh(range(len(order)), t3.hit_rate, xerr=[t3.hit_rate - t3.ci_low, t3.ci_high - t3.hit_rate], color=colors, height=0.6, capsize=3)
for y, v in enumerate(t3.hit_rate):
    ax.text(v + 0.02, y, f"{v:.1%}", va="center", fontsize=9)
ax.set(yticks=range(len(order)), yticklabels=order, xlim=(0, 1), xlabel="share of test career changers whose real next role is in the top 3",
       title="The right role in the top 3, on real career moves (held-out people)")
ax.invert_yaxis()
finish()

**What the data shows**
- **The anchor holds:** the trained layer from `ML_Fit`'s saved hand-off gives the same hit or miss as `ML_Fit` 6b for 100% of the 563 test people at top 3 and top 5 (81.7% [78.5%, 84.7%] at top 3, 96.3% at top 5), so the setup still reproduces `ML_Fit` exactly and the rankers are the only thing that moved.
- **No hypothesis is supported.** E1 `rrf` over the baseline +4.6 points (95% interval −0.9 to +10.0), E2 combined over the baseline +5.0 (+0.0 to +9.9, touching zero), E3 combined over `rrf` +0.4 (−2.7 to +3.4), E4 combined over `mlrank` **−1.2** (−4.4 to +2.0). By the 1c rule none of the four holds: on this answer key the three rankers are not separated from each other, and only the trained layer is clearly above the baseline.
- **The three rankers land within 1.5 points at top 3:** `mlrank` 69.4% [65.7%, 73.2%], combined 68.2% [64.5%, 72.1%], `rrf` 67.9% [63.9%, 71.6%], against the baseline's 63.2% [59.1%, 67.1%] and the trained layer's 81.7%. **At top 5 none of them beats the baseline** (87.0%): `mlrank` ties it, `rrf` 86.1%, combined 85.4%.
- **v1 scored 76.4% here and v2 scores 68.2%; the drop is the design change, not a regression.** v1 took the roles from the trained layer alone. v2 blends it equally with the role mix of the person's text matches, because on a real résumé the trained layer reads only titles, programs and dates and was leading with the wrong role (`JACOB.md`, 2026-10-16). The trained layer is the strongest role signal in the project (81.7%), so blending it toward the text match's 67.9% gives up most of v1's lead on this key — while v2's compensating half, a stated desired role ranked first, cannot appear at all, because Profiles has no desired-role field (`combined/evaluate.py`). v2 also caps the pool at the 30 best text matches per role, so the two versions differ in more than the blend and the 8.2-point gap has no single clean cause.
- **The ladder no longer rises:** text match alone 67.9% → plus ML_Rank's queue 69.4% → plus v2's role fit 68.2%, against the trained layer's 81.7%. The queue adds 1.5 points and v2's role fit gives back 1.2, neither interval excluding zero. The honest reading is that over whole-corpus listing rankings these three are alike, and the project's measured gain lives in the role model rather than in the ranker stack. The gap from the trained layer is also wider than before — 13.5 points against v1's 5.3 — so turning roles into listings is no longer the only thing between them.
- **By pool (top 3), v2 helps the people the project is for and hurts the ones who dominate the sample.** The combined model is the weakest of the three for moves within tech (66.0%, against `rrf`'s 68.9% and `mlrank`'s 68.7%) and the strongest for both retrained pools: 73.2% for people who retrained after working in another field (`mlrank` 69.5%, `rrf` 64.6%, baseline 57.3%), and 75.0% for new grads from another field, where it beats even the trained layer's 72.2%. Within-tech movers are 409 of the 563 test people, so they set the overall number. These two pools hold 82 and 72 people and this table carries no intervals, so that pattern is a direction worth testing, not a demonstrated difference.
- **Top 10 instead of 20 listings (secondary):** the same flat picture: `mlrank` 67.9%, combined 67.3%, `rrf` 65.9%, baseline 63.2% at top 3.
- **Speed:** the rankers load in 3.8 minutes over 193,043 serving listings (28,249 hidden by default); per person, `rrf` takes 0.4 seconds, `mlrank` 0.8 and the combined model 3.1 (four text-match calls plus the queue). These timings depend on what else the machine was running and are not a property of the rankers: `Evaluate`'s run had `mlrank` at 0.4 and the combined model at 1.5.

## 5. Conclusion

- **E1, E2, E3 and E4 all fail.** On real career moves, held-out people, the three rankers the app can serve are not separated from each other at top 3 — `mlrank` 69.4%, combined model (v2) 68.2%, `rrf` 67.9% — and the closest any comes to the 1c rule is combined's +5.0 points over the 63.2% baseline, on an interval that touches zero. At top 5 none of them beats the baseline.
- **The measured advantage of this project is the role model, not the ranker stack.** `ML_Fit`'s trained layer finds the role a career changer actually moved into for 81.7% [78.5%, 84.7%] of the same people. Everything the app does on top of a text match — the queue, v2's role fit — moves this metric by one or two points, inside its own noise.
- **v1 against v2 is a real trade-off, and only one side of it is measured.** v1 scored 76.4% here by taking roles from the trained layer alone; v2 blends in the text match's role mix so a real résumé leads with the right role, and always puts a stated desired role first. This answer key cannot see either fix: Profiles' backgrounds are the trained layer's own inputs and carry no desired role. So the drop to 68.2% is real but partial evidence, and deciding between the versions on this number alone would be deciding on the half that suits v1.
- **What it means for the demo** (`ANALYSIS.md`, Pitch): the "text match → plus the queue → plus role fit" ladder is not a rising staircase on v2 and should not be shown as one. What the evidence carries is the role result (81.7%, `ML_Fit` 6b) and the listing backtest (precision@10 0.500, `ML_Rank` 8e); this notebook's own contribution is the flat comparison and the reason it is flat.
- **Limits:** 8 roles only (Profiles has no moves into sales engineering or business and market research analysis); the backgrounds are résumé titles and programs from the public data, read with Profiles' rules, not an app user's résumé through the local reader, and they carry no desired role, so v2 is scored in its least favourable setting; this measures which roles a ranker points to, not which listings are good to apply to (that's `ML_Rank`'s backtest, 8d–8e); the per-pool pattern rests on 82 and 72 people with no intervals.
- **Next:** decide which version the app ships and what the pitch claims about the rankers (`ANALYSIS.md`, "Decide which combined-model version the app ships", with Jesse — his `app/APP.md` and `app/docs/ml-inventory.md` still quote v1's numbers). A fair test of v2 needs an answer key with desired roles and full résumé text, which Profiles cannot provide; until there is one, v2's advantage rests on the four résumé checks in `combined/tests/test_roles_from_profile.py`, not on held-out people.